In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from qdrant_client.models import VectorParams, Distance, PointStruct
from tqdm import tqdm
import os 
from FlagEmbedding import BGEM3FlagModel 

In [2]:
import torch, gc, os
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Prevents the most common deadlock
gc.collect()
torch.cuda.empty_cache()

In [3]:
import os
import torch

# Force clear any specific GPU selection variables
if "CUDA_VISIBLE_DEVICES" in os.environ:
    del os.environ["CUDA_VISIBLE_DEVICES"]

# Check if PyTorch can see the GPU again
print(f"Is CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Is CUDA available? True
GPU: NVIDIA GeForce RTX 3060


In [4]:
print("load bge m3")
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True,)


load bge m3


Fetching 30 files: 100%|██████████████████████| 30/30 [00:00<00:00, 3320.38it/s]


In [5]:
client = QdrantClient(url='http://localhost:6333')

In [6]:
collection_name = 'arxiv_papers'

In [2]:
csv_path  = "../arxiv_filtered_master.parquet"
df = pd.read_parquet(csv_path, columns = ['title','abstract','id','year'])

In [8]:
def hybrid_search(user_query, limit=5):
    query_output = model.encode(
        [user_query],
        return_dense = True,
        return_sparse=True
    )

    #use reciprocal rank fusion 
    results = client.query_points(
    collection_name = collection_name,
    prefetch=[
        models.Prefetch(
            query= query_output['dense_vecs'][0],
            using='dense',
            limit=20,
        ),
        models.Prefetch(
    query = models.SparseVector(
        indices = list(query_output['lexical_weights'][0].keys()),
        values = list(query_output['lexical_weights'][0].values())
    ),
    using='sparse',
    limit = 20
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=limit
    )

    print(f"\n🔍 Query: '{user_query}'")
    for i, hit in enumerate(results.points):
        print(f"{i+1}. [{hit.payload['id']}] {hit.payload['title']}")

In [9]:
hybrid_search("cognitive mirage")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



🔍 Query: 'cognitive mirage'
1. [2506.13990] Machine Mirages: Defining the Undefined
2. [2506.18998] Mirage of Mastery: Memorization Tricks LLMs into Artificially Inflated Self-Knowledge
3. [2505.10604] MIRAGE: A Multi-modal Benchmark for Spatial Perception, Reasoning, and Intelligence
4. [2508.02885] Merge-based syntax is mediated by distinct neurocognitive mechanisms: A clustering analysis of comprehension abilities in 84,000 individuals with language deficits across nine languages
5. [2511.14426] MiAD: Mirage Atom Diffusion for De Novo Crystal Generation


In [10]:
hybrid_search("low rank adaptation for large language models")



🔍 Query: 'low rank adaptation for large language models'
1. [2403.16187] ALoRA: Allocating Low-Rank Adaptation for Fine-tuning Large Language   Models
2. [2506.21408] Scalable Bayesian Low-Rank Adaptation of Large Language Models via Stochastic Variational Subspace Inference
3. [2410.04060] LoRTA: Low Rank Tensor Adaptation of Large Language Models
4. [2502.12171] GoRA: Gradient-driven Adaptive Low Rank Adaptation
5. [2409.17538] Low-Rank Adaptation Secretly Imitates Differentially Private SGD


In [11]:
hybrid_search("QLoRA")


🔍 Query: 'QLoRA'
1. [2507.04599] QR-LoRA: Efficient and Disentangled Fine-tuning via QR Decomposition for Customized Generation
2. [2305.14314] QLoRA: Efficient Finetuning of Quantized LLMs
3. [2410.14713] QuAILoRA: Quantization-Aware Initialization for LoRA
4. [2407.11046] A Survey on LoRA of Large Language Models
5. [2502.17920] C-LoRA: Continual Low-Rank Adaptation for Pre-trained Models


In [12]:
hybrid_search("2507.22781")


🔍 Query: '2507.22781'
1. [2301.07433] DDPEN: Trajectory Optimisation With Sub Goal Generation Model
2. [1306.3909] Copula-based Randomized Mechanisms for Truthful Scheduling on Two   Unrelated Machines
3. [2106.06706] Decentralized Matching in a Probabilistic Environment
4. [2507.14111] CUDA-L1: Improving CUDA Optimization via Contrastive Reinforcement Learning
5. [1503.00941] Approximation Algorithms for Computing Maximin Share Allocations


In [13]:
hybrid_search('original transformer mechanism paper')


🔍 Query: 'original transformer mechanism paper'
1. [2302.09327] Transformadores: Fundamentos teoricos y Aplicaciones
2. [2402.13572] AlgoFormer: An Efficient Transformer Framework with Algorithmic   Structures
3. [2306.07303] A Comprehensive Survey on Applications of Transformers for Deep Learning   Tasks
4. [2402.13388] Transformer tricks: Precomputing the first layer
5. [2306.00396] Lightweight Vision Transformer with Bidirectional Interaction


In [20]:
hybrid_search('optimize retrieval with there are many short documents in search of finding the most suitable documents')


🔍 Query: 'optimize retrieval with there are many short documents in search of finding the most suitable documents'
1. [1904.08375v2] Document Expansion by Query Prediction
2. [2504.16711v1] A Unified Retrieval Framework with Document Ranking and EDU Filtering
  for Multi-document Summarization
3. [2012.11685v2] Neural Methods for Effective, Efficient, and Exposure-Aware Information Retrieval
4. [1808.06528v1] Adaptive Document Retrieval for Deep Question Answering
5. [2504.16264v1] CLIRudit: Cross-Lingual Information Retrieval of Scientific Documents


In [15]:
df.index.str.contains('2305.14314').sum()

np.int64(0)

In [17]:
import hashlib
def stable_hash_to_int(string_id):
     hash_obj = hashlib.sha256(string_id.encode('utf-8'))
     return int(hash_obj.hexdigest(), 16) % (2**63) 

In [18]:
# Check if the "Attention" paper (1706.03762) and "QLoRA" (2305.14314) exist
ids_to_check = ["1706.03762", "2307.09288"]
print("--- Data Audit ---")
for pid in ids_to_check:
    # Hash it exactly as we did during indexing
    pid_int = stable_hash_to_int(pid + "v1") # Try v1
    # Note: You might need to check 'v2', 'v3' etc if you stripped versions, 
    # but let's check your hash logic.
    
    # Or better: Search by Payload (since we don't know the version you indexed)
    res = client.scroll(
        collection_name=collection_name,
        scroll_filter=models.Filter(
            must=[models.FieldCondition(key="arxiv_id", match=models.MatchText(text=pid))]
        ),
        limit=1
    )
    if res[0]:
        print(f"✅ {pid} is in the DB: {res[0][0].payload['title']}")
    else:
        print(f"❌ {pid} is NOT in the DB. (This is why search failed)")

--- Data Audit ---
✅ 1706.03762 is in the DB: Attention Is All You Need
❌ 2307.09288 is NOT in the DB. (This is why search failed)


In [4]:
df[df['title'].str.contains('cognitive mirage',case=False,na=False)]

,title,abstract,id,year
310447,Cognitive Mirage: A Review of Hallucinations i...,As large language models continue to develop i...,2309.06794,2023
